In [ ]:
# In[1]:

import pandas as pd

# Incident window bounds (UTC)
start_ts = pd.Timestamp("2024-01-20 04:43:33", tz="UTC")
end_ts = pd.Timestamp("2024-01-20 05:13:33", tz="UTC")

# --- metric.csv ---
metric_file = "metric.csv"
expected_metric_cols = {"timestamp", "cmdb_id", "kpi_name", "value"}
try:
    df_metric = pd.read_csv(metric_file)
except Exception as e:
    df_metric = None
    metric_error = f"Failed to read {metric_file}: {e}"

if df_metric is None:
    metric_summary = pd.DataFrame()
    metric_top = pd.DataFrame()
    metric_in_window_count = 0
    metric_earliest = None
    metric_latest = None
else:
    missing = expected_metric_cols - set(df_metric.columns)
    if missing:
        metric_missing = f"Missing columns in {metric_file}: {sorted(list(missing))}"
        metric_summary = pd.DataFrame()
        metric_top = pd.DataFrame()
        metric_in_window_count = 0
        metric_earliest = None
        metric_latest = None
    else:
        # parse timestamps as UTC
        df_metric["ts"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
        metric_earliest = df_metric["ts"].min()
        metric_latest = df_metric["ts"].max()
        # global aggregation (use full series)
        metric_grp = (
            df_metric.groupby(["cmdb_id", "kpi_name"])["value"]
            .agg(
                count="count",
                mean="mean",
                min="min",
                max="max",
                p50=lambda x: x.quantile(0.5),
                p90=lambda x: x.quantile(0.9),
                p95=lambda x: x.quantile(0.95),
            )
            .reset_index()
        )
        metric_top = metric_grp.sort_values("count", ascending=False).head(50)
        # incident window filtering for counting rows
        mask = (df_metric["ts"] >= start_ts) & (df_metric["ts"] <= end_ts)
        metric_in_window_count = int(mask.sum())

# --- trace.csv ---
trace_file = "trace.csv"
expected_trace_cols = {"timestamp", "cmdb_id", "trace_name", "value"}
try:
    df_trace = pd.read_csv(trace_file)
except Exception as e:
    df_trace = None
    trace_error = f"Failed to read {trace_file}: {e}"

if df_trace is None:
    trace_summary = pd.DataFrame()
    trace_top = pd.DataFrame()
    trace_in_window_count = 0
    trace_earliest = None
    trace_latest = None
else:
    missing = expected_trace_cols - set(df_trace.columns)
    if missing:
        trace_missing = f"Missing columns in {trace_file}: {sorted(list(missing))}"
        trace_summary = pd.DataFrame()
        trace_top = pd.DataFrame()
        trace_in_window_count = 0
        trace_earliest = None
        trace_latest = None
    else:
        df_trace["ts"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)
        trace_earliest = df_trace["ts"].min()
        trace_latest = df_trace["ts"].max()
        trace_grp = (
            df_trace.groupby(["cmdb_id", "trace_name"])["value"]
            .agg(
                count="count",
                mean="mean",
                min="min",
                max="max",
                p50=lambda x: x.quantile(0.5),
                p90=lambda x: x.quantile(0.9),
                p95=lambda x: x.quantile(0.95),
            )
            .reset_index()
        )
        trace_top = trace_grp.sort_values("count", ascending=False).head(50)
        mask = (df_trace["ts"] >= start_ts) & (df_trace["ts"] <= end_ts)
        trace_in_window_count = int(mask.sum())

# --- log.csv ---
log_file = "log.csv"
expected_log_cols = {"timestamp", "cmdb_id", "log_name", "value"}
try:
    df_log = pd.read_csv(log_file)
except Exception as e:
    df_log = None
    log_error = f"Failed to read {log_file}: {e}"

if df_log is None:
    log_summary = pd.DataFrame()
    log_top = pd.DataFrame()
    log_in_window_count = 0
    log_earliest = None
    log_latest = None
else:
    missing = expected_log_cols - set(df_log.columns)
    if missing:
        log_missing = f"Missing columns in {log_file}: {sorted(list(missing))}"
        log_summary = pd.DataFrame()
        log_top = pd.DataFrame()
        log_in_window_count = 0
        log_earliest = None
        log_latest = None
    else:
        df_log["ts"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)
        log_earliest = df_log["ts"].min()
        log_latest = df_log["ts"].max()
        log_grp = (
            df_log.groupby(["cmdb_id", "log_name"])["value"]
            .agg(
                count="count",
                mean="mean",
                min="min",
                max="max",
                p50=lambda x: x.quantile(0.5),
                p90=lambda x: x.quantile(0.9),
                p95=lambda x: x.quantile(0.95),
            )
            .reset_index()
        )
        log_top = log_grp.sort_values("count", ascending=False).head(50)
        mask = (df_log["ts"] >= start_ts) & (df_log["ts"] <= end_ts)
        log_in_window_count = int(mask.sum())

# --- error_logs.csv ---
err_file = "error_logs.csv"
expected_err_cols = {"timestamp", "cmdb_id", "message"}
try:
    df_err = pd.read_csv(err_file)
except Exception as e:
    df_err = None
    err_error = f"Failed to read {err_file}: {e}"

if df_err is None:
    err_total = 0
    err_distinct_cmdb = pd.Series([], dtype=object)
    err_in_window_count = 0
    err_missing = f"Could not read {err_file}"
else:
    missing = expected_err_cols - set(df_err.columns)
    if missing:
        err_missing = f"Missing columns in {err_file}: {sorted(list(missing))}"
        err_total = len(df_err)
        # provide what is available
        err_distinct_cmdb = pd.Series(sorted(df_err["cmdb_id"].dropna().unique())) if "cmdb_id" in df_err.columns else pd.Series([], dtype=object)
        if "timestamp" in df_err.columns:
            df_err["ts"] = pd.to_datetime(df_err["timestamp"], unit="s", utc=True)
            err_in_window_count = int(((df_err["ts"] >= start_ts) & (df_err["ts"] <= end_ts)).sum())
        else:
            err_in_window_count = 0
    else:
        df_err["ts"] = pd.to_datetime(df_err["timestamp"], unit="s", utc=True)
        err_total = len(df_err)
        err_distinct_cmdb = pd.Series(sorted(df_err["cmdb_id"].dropna().unique()))
        err_in_window_count = int(((df_err["ts"] >= start_ts) & (df_err["ts"] <= end_ts)).sum())

# Compact outputs: each summary table limited to top results and key scalars
# Expose variables (IPython will display them)
metric_top, metric_in_window_count, metric_earliest, metric_latest, trace_top, trace_in_window_count, trace_earliest, trace_latest, log_top, log_in_window_count, log_earliest, log_latest, err_total, err_distinct_cmdb, err_in_window_count

```
Out[1]:
```
summary = (
    "Telemetry summary for incident window 2024-01-20 04:43:33 to 2024-01-20 05:13:33 UTC:\n\n"
    "1) metric.csv:\n"
    "- Per-(cmdb_id, kpi_name) statistics were computed (top 50 by count are available).\n"
    "- Number of metric rows inside the incident window: 1875\n"
    f"- Earliest metric timestamp: 2024-01-20 04:46:00+00:00 UTC\n"
    f"- Latest metric timestamp: 2024-01-20 05:10:00+00:00 UTC\n"
    "- Example top KPI rows include: carts.cpu (count=25, mean≈1.624), carts.diskio, carts.mem, queue-master.diskio, rabbitmq.cpu, etc.\n\n"
    "2) trace.csv:\n"
    "- No trace series summary returned (aggregation empty). This indicates trace.csv had no usable trace series for aggregation.\n"
    "- Number of trace rows inside the incident window: 0\n"
    "- No earliest/latest trace timestamps available.\n\n"
    "3) log.csv:\n"
    "- Per-(cmdb_id, log_name) statistics were computed (top 50 by count are available).\n"
    "- Number of log rows inside the incident window: 446\n"
    f"- Earliest log timestamp: 2024-01-20 04:46:00+00:00 UTC\n"
    f"- Latest log timestamp: 2024-01-20 05:10:00+00:00 UTC\n"
    "- Notable entries: front-end.log.row_count (count=25, mean≈1793.44) and many log.error_count series which are zero across the sampled periods.\n\n"
    "4) error_logs.csv:\n"
    "- Total rows: 0\n"
    "- Distinct cmdb_id values: none\n"
    "- Number of error_log rows inside the incident window: 0\n\n"
    "Notes:\n"
    "- Global thresholds/percentiles were computed using the full available series for each metric/trace/log prior to any window filtering.\n"
    "- trace.csv contains no aggregated data for this dataset; confirm trace telemetry ingestion if traces are expected.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count          mean           min           max           p50           p90           p95
0          carts         cpu     25  1.624304e+00  1.488824e+00  2.424934e+00  1.553098e+00  1.710091e+00  1.985830e+00
1          carts      diskio     25  7.766288e+03  0.000000e+00  1.228164e+05  0.000000e+00  0.000000e+00  5.707265e+04
2          carts  latency-50     25  6.202986e-01  2.014821e-02  1.317460e+00  2.141462e-02  1.306824e+00  1.315465e+00
3          carts  latency-90     25  1.111925e+00  4.138143e-02  2.263492e+00  2.437419e-01  2.261365e+00  2.263093e+00
4          carts         mem     25  2.203910e+08  2.195397e+08  2.206263e+08  2.205226e+08  2.206102e+08  2.206196e+08
..           ...         ...    ...           ...           ...           ...           ...           ...           ...
45  queue-master      diskio     25  3.245909e+04  2.074786e+04  4.423068e+04  3.079572e+04  4.205800e+04  4.315340e+04
46  queue-master         mem     25  2.724048e+08  2.722934e+08  2.727419e+08  2.723040e+08  2.727133e+08  2.727144e+08
47  queue-master      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
48      rabbitmq         cpu     25  7.757435e-01  7.093987e-01  8.195058e-01  7.768302e-01  8.017633e-01  8.117444e-01
49      rabbitmq      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 9 columns], 1875, Timestamp('2024-01-20 04:46:00+0000', tz='UTC'), Timestamp('2024-01-20 05:10:00+0000', tz='UTC'), Empty DataFrame
Columns: [cmdb_id, trace_name, count, mean, min, max, p50, p90, p95]
Index: [], 0, NaT, NaT,          cmdb_id         log_name  count         mean  min   max     p50     p90     p95
7      front-end    log.row_count     25  1793.440000  758  2028  1866.0  2011.8  2022.6
6      front-end  log.error_count     25     0.000000    0     0     0.0     0.0     0.0
5      catalogue    log.row_count     25   150.520000   66   166   157.0   164.6   165.0
4      catalogue  log.error_count     25     0.000000    0     0     0.0     0.0     0.0
12  queue-master  log.error_count     25     0.000000    0     0     0.0     0.0     0.0
13  queue-master    log.row_count     25   502.920000  198   639   513.0   590.4   612.0
17      shipping    log.row_count     25    55.880000   22    71    57.0    65.6    68.0
16      shipping  log.error_count     25     0.000000    0     0     0.0     0.0     0.0
8         orders  log.error_count     25     0.000000    0     0     0.0     0.0     0.0
9         orders    log.row_count     25   111.760000   44   142   114.0   131.2   136.0
10       payment  log.error_count     25     0.000000    0     0     0.0     0.0     0.0
11       payment    log.row_count     25    94.280000   40   111    97.0   105.6   108.0
18          user  log.error_count     25     0.000000    0     0     0.0     0.0     0.0
19          user    log.row_count     25   542.160000  221   633   555.0   618.2   629.8
2       carts-db  log.error_count     24     0.000000    0     0     0.0     0.0     0.0
3       carts-db    log.row_count     24     1.000000    1     1     1.0     1.0     1.0
0          carts  log.error_count     19     0.000000    0     0     0.0     0.0     0.0
1          carts    log.row_count     19    64.473684   25   100    50.0   100.0   100.0
15    session-db    log.row_count      5     5.000000    5     5     5.0     5.0     5.0
14    session-db  log.error_count      5     0.000000    0     0     0.0     0.0     0.0, 446, Timestamp('2024-01-20 04:46:00+0000', tz='UTC'), Timestamp('2024-01-20 05:10:00+0000', tz='UTC'), 0, Series([], dtype: object), 0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Ensure metric data is loaded
if "df_metric" not in globals():
    df_metric = pd.read_csv("metric.csv")

# Parse timestamps as UTC-aware datetimes (use existing 'ts' if present)
if "ts" not in df_metric.columns:
    df_metric["ts"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# Incident window (UTC)
start_ts = pd.Timestamp("2024-01-20 04:43:33", tz="UTC")
end_ts = pd.Timestamp("2024-01-20 05:13:33", tz="UTC")

# Candidate services
candidates = ["carts", "catalogue", "orders", "payment", "user"]

# Filter to candidate cmdb_ids (global series - for threshold calculation)
df_candidates = df_metric[df_metric["cmdb_id"].isin(candidates)].copy()

# If no candidate rows, produce empty summary
if df_candidates.empty:
    anomaly_summary = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","global_count","global_mean","global_min","global_max","global_p95",
        "in_window_count","in_window_anomalies","earliest_anomaly_ts","in_window_max","ratio",
        "cmdb_earliest_ts","cmdb_latest_ts"
    ])
    notes = "No metric rows for the specified candidate services."
else:
    # 1) Global stats (P95 computed on full series)
    global_stats = (
        df_candidates.groupby(["cmdb_id", "kpi_name"])["value"]
        .agg(
            global_count="count",
            global_mean="mean",
            global_min="min",
            global_max="max",
            global_p95=lambda x: x.quantile(0.95),
        )
        .reset_index()
    )

    # Per-cmdb overall earliest/latest timestamps (for reporting when in-window rows = 0)
    cmdb_times = (
        df_candidates.groupby("cmdb_id")["ts"]
        .agg(cmdb_earliest_ts="min", cmdb_latest_ts="max")
        .reset_index()
    )

    # 2) Filter metric.csv to rows inside the incident window
    mask = (df_candidates["ts"] >= start_ts) & (df_candidates["ts"] <= end_ts)
    df_window = df_candidates[mask].copy()

    # 3) In-window aggregates per (cmdb_id, kpi_name)
    in_window_stats = (
        df_window.groupby(["cmdb_id", "kpi_name"])["value"]
        .agg(in_window_count="count", in_window_max="max")
        .reset_index()
    )

    # Attach global_p95 to in-window rows to mark anomalies (value >= global_p95)
    df_window = df_window.merge(global_stats[["cmdb_id", "kpi_name", "global_p95"]], on=["cmdb_id", "kpi_name"], how="left")
    df_window["is_anomaly"] = df_window["value"] >= df_window["global_p95"]

    # Count anomalies and earliest anomaly timestamp per group
    anomaly_stats = (
        df_window[df_window["is_anomaly"]]
        .groupby(["cmdb_id", "kpi_name"])
        .agg(in_window_anomalies=("is_anomaly", "sum"), earliest_anomaly_ts=("ts", "min"))
        .reset_index()
    )

    # Merge global_stats with in-window and anomaly stats (left join to keep all global groups)
    summary = global_stats.merge(in_window_stats, on=["cmdb_id", "kpi_name"], how="left")
    summary = summary.merge(anomaly_stats, on=["cmdb_id", "kpi_name"], how="left")
    summary = summary.merge(cmdb_times, on="cmdb_id", how="left")

    # Fill NaNs for groups with no in-window rows
    summary["in_window_count"] = summary["in_window_count"].fillna(0).astype(int)
    # in_window_max stays NaN if no rows in-window
    summary["in_window_anomalies"] = summary["in_window_anomalies"].fillna(0).astype(int)
    # earliest_anomaly_ts may be NaT; keep as datetime64[ns, UTC]
    summary["earliest_anomaly_ts"] = pd.to_datetime(summary["earliest_anomaly_ts"], utc=True)

    # Compute ratio = in_window_max / global_p95 with careful handling
    def compute_ratio(row):
        p95 = row["global_p95"]
        m = row["in_window_max"]
        if pd.isna(m):
            return np.nan
        # If p95 is zero:
        if pd.isna(p95) or p95 == 0:
            return np.inf if m > 0 else np.nan
        return m / p95

    summary["ratio"] = summary.apply(compute_ratio, axis=1)

    # Select and order columns for compact output
    cols = [
        "cmdb_id",
        "kpi_name",
        "global_count",
        "global_mean",
        "global_min",
        "global_max",
        "global_p95",
        "in_window_count",
        "in_window_anomalies",
        "earliest_anomaly_ts",
        "in_window_max",
        "ratio",
        "cmdb_earliest_ts",
        "cmdb_latest_ts",
    ]
    summary = summary[cols]

    # Order by anomalies desc then ratio desc
    summary = summary.sort_values(["in_window_anomalies", "ratio"], ascending=[False, False])

    # Limit to top 50 rows
    anomaly_summary = summary.head(50).reset_index(drop=True)

    # 4) Brief notes: most persistent (most anomalies, earliest timestamp tie-break) and most severe (highest ratio)
    if not anomaly_summary.empty and anomaly_summary["in_window_anomalies"].sum() > 0:
        persistent = anomaly_summary.sort_values(["in_window_anomalies", "earliest_anomaly_ts"], ascending=[False, True]).iloc[0]
        # For severe, treat inf as highest; use idxmax on ratio with nan-safe approach
        severe_idx = anomaly_summary["ratio"].replace([np.inf, -np.inf], np.finfo(float).max).fillna(-np.inf).idxmax()
        severe = anomaly_summary.loc[severe_idx]
        notes = (
            f"Most persistent anomaly: {persistent['cmdb_id']} / {persistent['kpi_name']} "
            f"(anomalies={int(persistent['in_window_anomalies'])}, earliest={persistent['earliest_anomaly_ts']})\n"
            f"Most severe anomaly by ratio: {severe['cmdb_id']} / {severe['kpi_name']} "
            f"(ratio={np.round(severe['ratio'],3) if pd.notna(severe['ratio']) else severe['ratio']}, "
            f"in_window_max={severe['in_window_max']}, P95={severe['global_p95']})"
        )
    else:
        notes = "No in-window anomalies detected for the candidate services."

# Output compact table (top <=50) and notes
anomaly_summary, notes

```
Out[2]:
```
Summary of metric-based anomaly detection for candidate services (carts, catalogue, orders, payment, user) during 2024-01-20 04:43:33–05:13:33 UTC:

- Most persistent anomaly:
  - catalogue / socket — every in-window point was anomalous (25 anomalies). Earliest anomaly: 2024-01-20 04:46:00 UTC.
- Most severe anomaly (largest exceedance over global P95):
  - carts / diskio — ratio = 2.152 (in-window max ≈ 122,816.38, global P95 ≈ 57,072.65). Earliest anomaly: 2024-01-20 04:46:00 UTC.
- Other notable findings:
  - payment / socket: 23 anomalies in-window; earliest anomaly 2024-01-20 04:46:00 UTC; ratio ≈ 1.0 (values at or just above P95).
  - user / socket: 6 anomalies; earliest anomaly 2024-01-20 04:46:00 UTC; ratio ≈ 1.0.
  - orders / diskio: 25 anomalies but the series values are all zero (global P95 = 0), so ratio is not meaningful (NaN).
  - Many mem and latency KPIs show small exceedances (ratios close to 1.0) and only a few in-window anomaly points.
- Global series counts and time span:
  - Most metric series for these components have global_count = 25.
  - Earliest metric timestamp for these components: 2024-01-20 04:46:00 UTC. Latest: 2024-01-20 05:10:00 UTC.

Interpretation / recommended next steps:
- Investigate carts: disk I/O shows the largest severity (over 2× P95) — check disk subsystem, I/O queueing, or workload causing spikes.
- Investigate catalogue: socket metric shows persistent anomalies across the entire window — check socket errors/connections for catalogue.
- Check payment and user for socket issues (multiple anomalous points).
- Treat orders.diskio cautiously: values are zero baseline (P95=0), so the zero anomalies are not actionable without different thresholding.

If you want, I can produce the compact table (top 50 rows) again or focus drill-down time series for the top offending KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name  global_count   global_mean    global_min    global_max    global_p95  in_window_count  in_window_anomalies       earliest_anomaly_ts  in_window_max     ratio          cmdb_earliest_ts            cmdb_latest_ts
0   catalogue      socket            25  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00               25                   25 2024-01-20 04:46:00+00:00   7.000000e+00  1.000000 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
1      orders      diskio            25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               25                   25 2024-01-20 04:46:00+00:00   0.000000e+00       NaN 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
2     payment      socket            25  5.947333e+00  5.000000e+00  6.000000e+00  6.000000e+00               25                   23 2024-01-20 04:46:00+00:00   6.000000e+00  1.000000 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
3        user      socket            25  1.912133e+01  1.800000e+01  2.000000e+01  2.000000e+01               25                    6 2024-01-20 04:46:00+00:00   2.000000e+01  1.000000 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
4       carts      diskio            25  7.766288e+03  0.000000e+00  1.228164e+05  5.707265e+04               25                    2 2024-01-20 04:46:00+00:00   1.228164e+05  2.151930 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
..        ...         ...           ...           ...           ...           ...           ...              ...                  ...                       ...            ...       ...                       ...                       ...
27       user         mem            25  8.616282e+06  8.592649e+06  8.665839e+06  8.654862e+06               25                    2 2024-01-20 05:08:00+00:00   8.665839e+06  1.001268 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
28  catalogue         mem            25  6.284392e+06  6.190080e+06  6.313165e+06  6.311649e+06               25                    2 2024-01-20 05:03:00+00:00   6.313165e+06  1.000240 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
29      carts  latency-90            25  1.111925e+00  4.138143e-02  2.263492e+00  2.263093e+00               25                    2 2024-01-20 05:04:00+00:00   2.263492e+00  1.000176 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
30     orders         mem            25  3.191389e+08  3.188413e+08  3.193637e+08  3.193456e+08               25                    2 2024-01-20 04:50:00+00:00   3.193637e+08  1.000057 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00
31      carts         mem            25  2.203910e+08  2.195397e+08  2.206263e+08  2.206196e+08               25                    2 2024-01-20 05:01:00+00:00   2.206263e+08  1.000031 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00

[32 rows x 14 columns], 'Most persistent anomaly: catalogue / socket (anomalies=25, earliest=2024-01-20 04:46:00+00:00)\nMost severe anomaly by ratio: carts / diskio (ratio=2.152, in_window_max=122816.37894048808, P95=57072.654140276514)')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Ensure dataframes are loaded (reuse if present)
if "df_metric" not in globals():
    df_metric = pd.read_csv("metric.csv")
if "df_log" not in globals():
    df_log = pd.read_csv("log.csv")
if "df_err" not in globals():
    try:
        df_err = pd.read_csv("error_logs.csv")
    except Exception:
        df_err = pd.DataFrame()  # empty if missing

# Parse timestamps to UTC-aware datetimes if not already
if "ts" not in df_metric.columns:
    df_metric["ts"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
if "ts" not in df_log.columns:
    df_log["ts"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)
if not df_err.empty and "ts" not in df_err.columns:
    df_err["ts"] = pd.to_datetime(df_err["timestamp"], unit="s", utc=True)

# Incident window (UTC)
start_ts = pd.Timestamp("2024-01-20 04:43:33", tz="UTC")
end_ts = pd.Timestamp("2024-01-20 05:13:33", tz="UTC")

# Candidates (specific KPI pairs)
candidates = [
    ("catalogue", "socket"),
    ("carts", "diskio"),
    ("payment", "socket"),
    ("user", "socket"),
    ("orders", "diskio"),
]

# 1) Compute global P95 for each candidate using full metric series (global thresholds)
# Filter global series for candidate cmdb_id/kpi_name combinations
candidate_df = df_metric[df_metric["cmdb_id"].isin({c for c, k in candidates})].copy()

# Compute global stats for all (cmdb_id, kpi_name) present in candidate list
global_stats = (
    candidate_df.groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(global_count="count",
         global_mean="mean",
         global_min="min",
         global_max="max",
         global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Keep only the requested candidate pairs (even if some pairs missing from data, include rows for them later)
global_stats = global_stats.set_index(["cmdb_id", "kpi_name"]).reindex(pd.MultiIndex.from_tuples(candidates, names=["cmdb_id","kpi_name"])).reset_index()

# 2) Filter metric rows to incident window
mask_window = (df_metric["ts"] >= start_ts) & (df_metric["ts"] <= end_ts)
df_metric_window = df_metric[mask_window].copy()

# 3) For each candidate produce consecutive anomalous segments
segments = []

# Helper: expected consecutive delta (one minute)
one_min = pd.Timedelta(minutes=1)

for cmdb_id, kpi_name in candidates:
    # global p95 for this pair
    row = global_stats[(global_stats["cmdb_id"] == cmdb_id) & (global_stats["kpi_name"] == kpi_name)]
    if row.empty or pd.isna(row.iloc[0]["global_p95"]):
        global_p95 = np.nan
        global_count = int(row.iloc[0]["global_count"]) if (not row.empty and not pd.isna(row.iloc[0]["global_count"])) else 0
    else:
        global_p95 = float(row.iloc[0]["global_p95"])
        global_count = int(row.iloc[0]["global_count"]) if not pd.isna(row.iloc[0]["global_count"]) else 0

    # Extract in-window series for this candidate
    s = df_metric_window[(df_metric_window["cmdb_id"] == cmdb_id) & (df_metric_window["kpi_name"] == kpi_name)].copy().sort_values("ts")
    if s.empty:
        # Include zero in-window case per instruction later (we won't produce segments)
        continue

    # Determine anomaly condition per rule:
    # if global_p95 == 0 -> anomalous when value > 0
    # else anomalous when value >= global_p95
    if not pd.isna(global_p95) and global_p95 == 0:
        s["is_anomaly"] = s["value"] > 0
    else:
        s["is_anomaly"] = s["value"] >= global_p95

    # Keep only anomalous points
    s_anom = s[s["is_anomaly"]].copy()
    if s_anom.empty:
        continue

    # Identify consecutive segments: consecutive timestamps with difference == 1 minute
    s_anom = s_anom.sort_values("ts").reset_index(drop=True)
    # compute time diff from previous point
    s_anom["prev_ts"] = s_anom["ts"].shift(1)
    s_anom["delta_prev"] = s_anom["ts"] - s_anom["prev_ts"]
    # start of segment when delta_prev != one_min or prev_ts is NaT
    s_anom["segment_start_flag"] = (s_anom["delta_prev"] != one_min) | s_anom["prev_ts"].isna()
    s_anom["segment_id"] = s_anom["segment_start_flag"].cumsum()

    # Aggregate segments
    seg_agg = s_anom.groupby("segment_id").agg(
        cmdb_id=("cmdb_id", "first"),
        kpi_name=("kpi_name", "first"),
        segment_start_ts=("ts", "min"),
        segment_end_ts=("ts", "max"),
        points_in_segment=("ts", "count"),
        segment_max_value=("value", "max"),
    ).reset_index(drop=True)

    # Attach global_p95 and ratio (NaN if p95==0 or p95 is NaN)
    seg_agg["global_p95"] = global_p95
    seg_agg["global_count"] = global_count
    seg_agg["ratio"] = seg_agg["segment_max_value"] / global_p95 if (not pd.isna(global_p95) and global_p95 != 0) else np.nan

    # Exclude single-point segments (points_in_segment >= 2)
    seg_agg = seg_agg[seg_agg["points_in_segment"] >= 2]

    # Append to list
    if not seg_agg.empty:
        segments.append(seg_agg)

# Combine all segments
if segments:
    segments_df = pd.concat(segments, ignore_index=True)
    # Order by points_in_segment desc then ratio desc
    segments_df = segments_df.sort_values(["points_in_segment", "ratio"], ascending=[False, False]).reset_index(drop=True)
    # Limit to top 20 rows and select requested columns
    segments_df = segments_df[["cmdb_id", "kpi_name", "segment_start_ts", "segment_end_ts", "points_in_segment", "segment_max_value", "global_p95", "ratio"]].head(20)
else:
    segments_df = pd.DataFrame(columns=["cmdb_id","kpi_name","segment_start_ts","segment_end_ts","points_in_segment","segment_max_value","global_p95","ratio"])

# 2) Compact log inspection for candidate cmdb_ids
candidate_cmdbs = ["catalogue", "carts", "payment", "user", "orders"]
df_log_window = df_log[(df_log["cmdb_id"].isin(candidate_cmdbs)) & (df_log["ts"] >= start_ts) & (df_log["ts"] <= end_ts)].copy()

if df_log_window.empty:
    log_agg = pd.DataFrame(columns=["cmdb_id","log_name","count","mean","min","max"])
    log_anom_rows = pd.DataFrame(columns=["timestamp","cmdb_id","log_name","value"])
else:
    # Aggregation per (cmdb_id, log_name)
    log_agg = df_log_window.groupby(["cmdb_id", "log_name"])["value"].agg(count="count", mean="mean", min="min", max="max").reset_index()
    # Limit to top 50 by count desc
    log_agg = log_agg.sort_values("count", ascending=False).head(50)

    # Rows where log_name contains 'error' (case-insensitive) OR value != 0
    cond_error_name = df_log_window["log_name"].str.contains("error", case=False, na=False)
    cond_value_nonzero = df_log_window["value"] != 0
    log_anom = df_log_window[cond_error_name | cond_value_nonzero].copy()
    # Select up to first 10 rows ordered by timestamp ascending
    if not log_anom.empty:
        log_anom_rows = log_anom.sort_values("ts")[["ts", "cmdb_id", "log_name", "value"]].head(10).rename(columns={"ts":"timestamp"})
    else:
        log_anom_rows = pd.DataFrame(columns=["timestamp","cmdb_id","log_name","value"])

# 3) Check error_logs.csv for these cmdb_ids in-window
if df_err.empty:
    err_in_window_df = pd.DataFrame(columns=["timestamp","cmdb_id","message"])
    err_note = "error_logs.csv missing or empty"
else:
    err_mask = (df_err["cmdb_id"].isin(candidate_cmdbs)) & (df_err["ts"] >= start_ts) & (df_err["ts"] <= end_ts)
    err_in_window = df_err[err_mask].copy()
    if err_in_window.empty:
        err_in_window_df = pd.DataFrame(columns=["timestamp","cmdb_id","message"])
    else:
        err_in_window_df = err_in_window.sort_values("ts")[["ts","cmdb_id","message"]].head(10).rename(columns={"ts":"timestamp"})

# One-line summary note: which candidate KPI(s) have multi-minute faults and whether logs corroborate
if segments_df.empty:
    summary_note = "No multi-point anomalous segments found for the candidate KPIs; logs show no corroborating error/non-zero entries for these services in-window." 
else:
    # Identify top multi-minute KPIs by points_in_segment and ratio
    top_multi = segments_df.sort_values(["points_in_segment","ratio"], ascending=[False,False]).iloc[0]
    # Check if logs show anomalies for that cmdb_id
    cmdb_logs = log_anom_rows[log_anom_rows["cmdb_id"] == top_multi["cmdb_id"]]
    corroboration = "with" if not cmdb_logs.empty else "without"
    summary_note = (
        f"Top multi-minute fault: {top_multi['cmdb_id']} / {top_multi['kpi_name']} "
        f"(points={int(top_multi['points_in_segment'])}, start={pd.to_datetime(top_multi['segment_start_ts'], utc=True)}) — logs {corroboration} corroborating error/non-zero entries."
    )

# Final compact outputs
segments_df, log_agg, log_anom_rows, err_in_window_df, summary_note

```
Out[3]:
```
Summary of focused checks (incident window 2024-01-20 04:43:33–05:13:33 UTC):

1) Consecutive-anomaly metric segments (multi-point segments only)
- catalogue / socket: continuous anomaly 04:46 → 05:10 (25 data points), segment_max=7, global_P95=7, ratio=1.00 — persistent multi-minute socket anomaly.
- payment / socket: continuous anomaly 04:46 → 05:08 (23 data points), segment_max=6, global_P95=6, ratio=1.00 — persistent multi-minute socket anomaly.
- user / socket: continuous anomaly 04:46 → 04:51 (6 data points), segment_max=20, global_P95=20, ratio=1.00 — multi-minute socket anomaly.
- carts / diskio: short multi-point segment 04:46 → 04:47 (2 data points), segment_max≈122,816.38, global_P95≈57,072.65, ratio≈2.152 — brief but severe disk I/O spike.
- orders / diskio: no multi-point anomalous segment found (no segments with >=2 consecutive anomalous points).

2) Compact log inspection (in-window)
- Aggregated in-window log stats (examples):
  - catalogue.log.row_count: count=25, mean≈150.52, min=66, max=166
  - catalogue.log.error_count: count=25, mean=0, all zeros
  - payment.log.row_count: count=25, mean≈94.28, min=40, max=111
  - payment.log.error_count: all zeros
  - orders.log.row_count: count=25, mean≈111.76, min=44, max=142
  - orders.log.error_count: all zeros
  - user.log.row_count: count=25, mean≈542.16, min=221, max=633
  - user.log.error_count: all zeros
  - carts.log.row_count: count=19, mean≈64.47, min=25, max=100
  - carts.log.error_count: all zeros

- Sample anomalous log rows (first up to 10 in-window where log_name contains "error" or value != 0), ordered by time:
  - 2024-01-20 04:46:00 — catalogue — log.row_count = 66
  - 2024-01-20 04:46:00 — payment  — log.row_count = 40
  - 2024-01-20 04:46:00 — orders   — log.row_count = 44
  - 2024-01-20 04:46:00 — user     — log.row_count = 221
  - (many log.error_count rows are present but their value = 0)

3) error_logs.csv check
- No error_log rows found for catalogue, carts, payment, user, or orders within the incident window.

One-line conclusion
- catalogue/socket and payment/socket show sustained multi-minute socket anomalies (catalogue is the longest), user/socket shows a shorter multi-minute anomaly, and carts/diskio shows a short but high-severity I/O spike (≈2.15× P95). Logs show elevated row_count activity for these services but log.error_count entries are zero and error_logs contain no records in-window — metrics indicate socket/disk issues while logs do not show explicit error messages corroborating them.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id kpi_name          segment_start_ts            segment_end_ts  points_in_segment  segment_max_value   global_p95    ratio
0  catalogue   socket 2024-01-20 04:46:00+00:00 2024-01-20 05:10:00+00:00                 25            7.00000      7.00000  1.00000
1    payment   socket 2024-01-20 04:46:00+00:00 2024-01-20 05:08:00+00:00                 23            6.00000      6.00000  1.00000
2       user   socket 2024-01-20 04:46:00+00:00 2024-01-20 04:51:00+00:00                  6           20.00000     20.00000  1.00000
3      carts   diskio 2024-01-20 04:46:00+00:00 2024-01-20 04:47:00+00:00                  2       122816.37894  57072.65414  2.15193,      cmdb_id         log_name  count        mean  min  max
3  catalogue    log.row_count     25  150.520000   66  166
2  catalogue  log.error_count     25    0.000000    0    0
6    payment  log.error_count     25    0.000000    0    0
7    payment    log.row_count     25   94.280000   40  111
4     orders  log.error_count     25    0.000000    0    0
5     orders    log.row_count     25  111.760000   44  142
8       user  log.error_count     25    0.000000    0    0
9       user    log.row_count     25  542.160000  221  633
1      carts    log.row_count     19   64.473684   25  100
0      carts  log.error_count     19    0.000000    0    0,                    timestamp    cmdb_id         log_name  value
2  2024-01-20 04:46:00+00:00  catalogue  log.error_count      0
3  2024-01-20 04:46:00+00:00  catalogue    log.row_count     66
6  2024-01-20 04:46:00+00:00     orders  log.error_count      0
7  2024-01-20 04:46:00+00:00     orders    log.row_count     44
8  2024-01-20 04:46:00+00:00    payment  log.error_count      0
9  2024-01-20 04:46:00+00:00    payment    log.row_count     40
14 2024-01-20 04:46:00+00:00       user  log.error_count      0
15 2024-01-20 04:46:00+00:00       user    log.row_count    221
30 2024-01-20 04:47:00+00:00       user  log.error_count      0
25 2024-01-20 04:47:00+00:00    payment    log.row_count     98, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 'Top multi-minute fault: catalogue / socket (points=25, start=2024-01-20 04:46:00+00:00) — logs with corroborating error/non-zero entries.')```
```